# Meta DE System Design — 60-Minute Whiteboard Walkthrough

The Meta onsite product-sense + full-stack round. 60 minutes. One product goal. Define the metrics, design the schema, write the ETL SQL, and produce a cost model. The hardest round in the Meta DE loop per DataDriven 2026.

The 5-step framework:

1. Product goal -> 2-3 success metrics.
2. Schema that supports the metrics (fact + dims, named grain).
3. ETL SQL that loads the schema from raw event tables.
4. Cost model (storage, compute, egress per 1M events).
5. Failure modes (idempotency, late events, schema drift).

Worked example below: 'Build a real-time dashboard for the WhatsApp Business Messaging team, surfacing per-business message volume and 7-day retention.'

In [1]:
import os, sqlite3
BASE = os.path.abspath(os.path.join(os.getcwd(), '..')) if 'notebooks' in os.getcwd() else os.getcwd()
conn = sqlite3.connect(':memory:')
with open(os.path.join(BASE, 'code', 'meta_schema.sql')) as f:
    conn.executescript(f.read())
print('schema loaded')

schema loaded


## Step 1 — Product goal + 2-3 metrics

- Goal: 'Surface per-business message volume and 7-day retention for the WA Business team.'
- Metrics:
  - Daily active businesses (DAB): distinct business_id with >=1 message sent.
  - 7-day retention: % of businesses active on day D who sent at least 1 message in the next 7 days.
  - Power-business rate: % of businesses sending >=100 messages per week.

## Step 2 — Schema (fact + dims, named grain)

Fact: `fct_business_message_daily` — one row per (business_id, day). Grain explicit.

Dims: `dim_business` (SCD2 on `business_size` and `industry`), `dim_date`.

## Step 3 — ETL SQL (the warehouse load)

Below: a simplified ETL that rolls the `whatsapp_message` raw event into the daily fact. Idempotency: `INSERT OR REPLACE` keyed on (business_id, day).

In [2]:
etl = """
CREATE TABLE fct_business_message_daily (
    business_id INTEGER,
    day         TEXT,
    n_messages  INTEGER,
    n_unique_receivers INTEGER,
    PRIMARY KEY (business_id, day)
);
INSERT OR REPLACE INTO fct_business_message_daily
SELECT sender_id AS business_id,
       DATE(sent_ts) AS day,
       COUNT(*) AS n_messages,
       COUNT(DISTINCT receiver_id) AS n_unique_receivers
FROM   whatsapp_message
WHERE  DATE(sent_ts) >= DATE('now', '-30 days')
GROUP BY sender_id, DATE(sent_ts);
"""
conn.executescript(etl)
for row in conn.execute("SELECT * FROM fct_business_message_daily ORDER BY day, business_id"):
    print(row)

## Step 4 — Cost model (per 1M events)

- Storage: 1KB/event, Parquet, 1M events = 1GB. S3 standard = $23/TB/mo. So $0.023 per 1M events per month.
- Compute: 1 vCPU-hour handles ~5M events in Spark. $0.05/vCPU-hr on EMR. So $0.01 per 1M events.
- Egress: dashboard serves 10K queries/mo at 1MB each = 10GB. $0.09/GB. So $0.90/mo total egress.
- **Total: ~$1/mo per 1M events ingested.** That is the order-of-magnitude answer.

The senior move is to name the 10x projection: at 1B events/day, the storage layer still costs $700/mo but the compute layer is the bottleneck, and you need a streaming pipeline (Kafka + Flink) rather than a daily batch.